# LLM Smoke Test

Use this notebook to verify a single OpenRouter-backed LLM call outside the FastAPI pipeline.

Before running:
- ensure `backend/.env` contains `OPENAI_API_KEY`, `OPENAI_API_BASE`, and `OPENROUTER_MODEL`
- activate the backend environment or open this notebook from the `backend/` directory

In [1]:
from pathlib import Path
import os
import sys

backend_root = Path.cwd()
if backend_root.name == "notebooks":
    backend_root = backend_root.parent

if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

backend_root

WindowsPath('d:/Developement/contract-analyzer/backend')

In [2]:
from app.core.settings import get_settings

get_settings.cache_clear()
settings = get_settings()

print({
    "model": settings.openrouter_model,
    "base_url": settings.openai_api_base,
    "timeout_seconds": settings.openai_timeout_seconds,
    "max_retries": settings.openai_max_retries,
})

{'model': 'google/gemma-4-26b-a4b-it:free', 'base_url': 'https://openrouter.ai/api/v1', 'timeout_seconds': 90.0, 'max_retries': 0}


In [3]:
from app.core.llm import get_llm

llm = get_llm(temperature=0)
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002682EFED040>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002682F2BD850>, root_client=<openai.OpenAI object at 0x000002682E9A3FB0>, root_async_client=<openai.AsyncOpenAI object at 0x000002682EAAAAE0>, model_name='google/gemma-4-26b-a4b-it:free', temperature=0.0, model_kwargs={'response_format': {'type': 'json_object'}}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', openai_proxy=None, request_timeout=90.0, max_retries=0, reasoning={'effort': 'none'}, stream_chunk_timeout=120.0)

In [4]:
messages = [
    {
        "role": "system",
        "content": "Return only valid JSON with a single key named status."
    },
    {
        "role": "user",
        "content": "Respond with {\"status\": \"ok\"}."
    },
]

messages

[{'role': 'system',
  'content': 'Return only valid JSON with a single key named status.'},
 {'role': 'user', 'content': 'Respond with {"status": "ok"}.'}]

In [6]:
import asyncio
import json

async def run_smoke_test():
    try:
        response = await llm.ainvoke(messages)
        content = response.content if isinstance(response.content, str) else str(response.content)
        print("Raw content:\n", content)
        parsed = json.loads(content)
        print("\nParsed JSON:\n", parsed)
        print("\nUsage metadata:\n", getattr(response, "usage_metadata", None))
        print("\nResponse metadata:\n", getattr(response, "response_metadata", None))
    except Exception as exc:
        print(type(exc).__name__)
        print(exc)
        raise

await run_smoke_test()

RateLimitError
Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-26b-a4b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3DtWSlGzNp8xUlpTd7xJtIgGYfO'}


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-26b-a4b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3DtWSlGzNp8xUlpTd7xJtIgGYfO'}

## Notes

- If this single call fails with `429`, the problem is provider capacity or model selection, not your chunking logic.
- If this succeeds but `/api/v1/analyze` fails, the problem is likely the multi-chunk segmentation flow.
- Change `OPENROUTER_MODEL` in `backend/.env` and re-run to compare models.